In [1]:
"""
Data Collection Script — Ghana Fuel Price Forecasting Project (Optimised)
Collects: Brent Crude Oil, GHS/USD Exchange Rate (Monthly Averages)
Sources:  EIA API, Frankfurter API
"""

import requests
import pandas as pd
from datetime import datetime
import time

# ─────────────────────────────────────────────
# CONFIGURATION (Updated for Post-Deregulation Window)
# ─────────────────────────────────────────────
START_DATE = "2015-06-01"   # Post-deregulation launch
END_DATE   = "2026-05-31"   # Capturing up to the latest full month
EIA_API_KEY = "5L8SAT2zgbtGXdX2E3gadXuj7cpQPTPhVJBbyMOz" 


# ─────────────────────────────────────────────
# 1. BRENT CRUDE OIL PRICES — EIA API
# ─────────────────────────────────────────────
def get_brent_crude(api_key, start, end):
    print("Fetching Brent Crude Oil prices from EIA...")
    url = (
        f"https://api.eia.gov/v2/petroleum/pri/spt/data/"
        f"?api_key={api_key}"
        f"&frequency=monthly"
        f"&data[0]=value"
        f"&facets[series][]=RBRTE"
        f"&start={start[:7]}"
        f"&end={end[:7]}"
        f"&sort[0][column]=period"
        f"&sort[0][direction]=asc"
        f"&offset=0&length=5000"
    )
    response = requests.get(url)
    if response.status_code != 200:
        print(f"  ERROR: {response.status_code} - {response.text}")
        return None

    data = response.json()
    records = data.get("response", {}).get("data", [])
    if not records:
        print("  No data returned. Check API key or date range.")
        return None

    df = pd.DataFrame(records)[["period", "value"]]
    df.columns = ["date", "brent_crude_usd"]
    df["date"] = pd.to_datetime(df["date"])
    df["brent_crude_usd"] = pd.to_numeric(df["brent_crude_usd"], errors="coerce")
    print(f"  Retrieved {len(df)} monthly observations.")
    return df


# ─────────────────────────────────────────────
# 2. GHS/USD EXCHANGE RATE — Frankfurter API (Corrected to Mean)
# ─────────────────────────────────────────────
def get_exchange_rate(start, end):
    print("Fetching GHS/USD exchange rate from Frankfurter...")
    records = []
    
    url = f"https://api.frankfurter.app/{start}..{end}?from=USD&to=GHS"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"  ERROR: {response.status_code} - {response.text}")
        return None

    data = response.json()
    rates = data.get("rates", {})

    for date_str, currency_dict in rates.items():
        ghs_rate = currency_dict.get("GHS")
        if ghs_rate:
            records.append({"date": date_str, "exchange_rate_ghs_usd": ghs_rate})

    if not records:
        print("  No data returned.")
        return None

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")

    # CRITICAL FIX: Changed from .last() to .mean() to get the true Monthly Average
    df_monthly = df.resample("MS").mean().reset_index()
    print(f"  Retrieved {len(df_monthly)} monthly averages.")
    return df_monthly


# ─────────────────────────────────────────────
# 3. MERGE AUTOMATED DATASETS
# ─────────────────────────────────────────────
def merge_datasets(brent_df, fx_df):
    print("\nMerging automated datasets...")

    for df in [brent_df, fx_df]:
        df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp()

    merged = brent_df.merge(fx_df, on="date", how="outer")
    merged = merged.sort_values("date").reset_index(drop=True)

    # Filter to exact study window
    merged = merged[
        (merged["date"] >= START_DATE) &
        (merged["date"] <= END_DATE)
    ]

    print(f"  API Pipeline complete: {len(merged)} rows recovered.")
    return merged


if __name__ == "__main__":
    print("=" * 55)
    print("   Ghana Fuel Price Pipeline — External Variables")
    print("=" * 55)

    brent_df = get_brent_crude(EIA_API_KEY, START_DATE, END_DATE)
    time.sleep(1)
    fx_df = get_exchange_rate(START_DATE, END_DATE)

    if brent_df is not None and fx_df is not None:
        final_df = merge_datasets(brent_df, fx_df)
        output_path = "api_external_data.csv"
        final_df.to_csv(output_path, index=False)
        print(f"\n✓ Base automated data saved to: {output_path}")
    else:
        print("\n✗ API execution failed.")

    print("\n" + "!"*40)
    print("REQUIRED NEXT MANUAL STEPS:")
    print("1. Download the actual MONTHLY CPI series from GSS StatsBank.")
    print("2. Pull the 2015-2026 Fuel Prices (Petrol, Diesel, LPG) from GSS/NPA.")
    print("3. Join those columns to 'api_external_data.csv' using matching dates.")
    print("!"*40)

   Ghana Fuel Price Pipeline — External Variables
Fetching Brent Crude Oil prices from EIA...
  Retrieved 132 monthly observations.
Fetching GHS/USD exchange rate from Frankfurter...
  ERROR: 404 - {"message":"not found"}

✗ API execution failed.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
REQUIRED NEXT MANUAL STEPS:
1. Download the actual MONTHLY CPI series from GSS StatsBank.
2. Pull the 2015-2026 Fuel Prices (Petrol, Diesel, LPG) from GSS/NPA.
3. Join those columns to 'api_external_data.csv' using matching dates.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


In [2]:
external_data.head()

NameError: name 'external_data' is not defined